# Validation Error Analysis

Review validation-only diagnostics for the strongest non-dummy model families from a completed ablation run. This notebook intentionally uses `features_val.csv` and ablation validation predictions only; do not point it at the held-out test table.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.models.error_analysis import run_validation_error_analysis

RUN_DIR = Path("outputs/runs/full_ablation_20260718")
VALIDATION_FEATURES = Path("data/processed/features_val.csv")
FEATURE_MANIFEST = Path("data/processed/feature_manifest.csv")
EPOCH_INDEX = Path("data/interim/epoch_index.csv")

## Build Or Refresh Analysis Artifacts

Set `allow_partial=True` only for a smoke pass while an ablation run is still in progress. For final validation analysis, keep it `False` so missing ablation/model outputs fail loudly.

In [ ]:
outputs = run_validation_error_analysis(
    run_dir=RUN_DIR,
    validation_features_path=VALIDATION_FEATURES,
    manifest_path=FEATURE_MANIFEST,
    epoch_index_path=EPOCH_INDEX,
    allow_partial=False,
    compute_permutation=True,
    permutation_repeats=5,
    compute_shap=True,
    max_shap_rows=1000,
    create_plots=True,
)
outputs.analysis_dir

## Selected Models

In [ ]:
selected_models = pd.read_csv(outputs.selected_models_path)
selected_models[[
    "ablation",
    "model",
    "n_features",
    "best_cv_macro_f1",
    "validation_macro_f1",
    "validation_accuracy",
]]

## Cross-Ablation Summary

In [ ]:
display(Image(filename=str(outputs.figures_dir / "model_family_macro_f1.png")))
pd.read_csv(outputs.model_family_comparison_path)

## Per-Class Metrics

In [ ]:
per_class = pd.read_csv(outputs.per_class_metrics_path)
per_class.sort_values(["ablation", "model", "label"])

## Diagnostic Figures By Ablation And Model

In [ ]:
artifact_index = pd.read_csv(outputs.artifact_index_path)
figure_index = artifact_index[artifact_index["artifact_type"] == "figure"].copy()

for row in selected_models.itertuples(index=False):
    print(f"\n=== {row.ablation} / {row.model} ===")
    subset = figure_index[
        (figure_index["ablation"] == row.ablation)
        & (figure_index["model"] == row.model)
    ]
    for figure_path in subset["path"]:
        print(Path(figure_path).name)
        display(Image(filename=figure_path))

## Participant And Transition Tables

In [ ]:
participant_metrics = pd.read_csv(outputs.per_participant_metrics_path)
transition_metrics = pd.read_csv(outputs.transition_metrics_path)

display(participant_metrics.sort_values("macro_f1").head(20))
display(transition_metrics)

## Feature Importance

In [ ]:
feature_importance = pd.read_csv(outputs.feature_importance_path)
display(feature_importance.head(50))

if outputs.permutation_importance_path is not None:
    permutation_importance = pd.read_csv(outputs.permutation_importance_path)
    display(permutation_importance.head(50))